# Part 2 – Python: Building a Reproducible Traffic Analytics Pipeline

## Task 1 – Data Pipeline Construction

This section develops and tests the data pipeline that will later be consolidated
into `pipeline.py`.

Task 1 includes:
1. Loading the raw CSV file with exception handling
2. Validating the source data schema
3. Standardising categorical values
4. Parsing and validating date/time values
5. Detecting and removing duplicate rows
6. Detecting and handling impossible values and outliers
7. Applying appropriate imputation where required
8. Using conditionals and loops where appropriate
9. Logging all important pipeline events, warnings, and errors

### Task 1.1 – Environment and Working Directory

The working directory is checked before running the pipeline to ensure that
project files are created and accessed from the correct GitHub repository folder.

In [1]:
import os

os.getcwd()

'C:\\Users\\tanak\\Documents\\GitHub\\smart-city-traffic-capstone\\part2_python'

### Task 1.2 – Logging Configuration

Python logging is configured to provide a traceable record of pipeline activity.

INFO messages are displayed in the console, while DEBUG and higher-level
messages are written to `pipeline.log`. The log format includes the timestamp,
log level, module name, and message.

In [2]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path

In [3]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

In [4]:
# Avoid adding duplicate handlers when this cell is run more than once
if not logger.handlers:
    formatter = logging.Formatter(
        "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
    )

    # Console output
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    # File output
    file_handler = logging.FileHandler("pipeline.log", mode="a")
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

In [5]:
logger.info("Logging configuration completed successfully.")

2026-10-06 05:13:12,833 - INFO - __main__ - Logging configuration completed successfully.


### Task 1.3 – Load the Raw CSV File

The raw Metro Interstate Traffic Volume CSV file is accessed using a relative
path so that the pipeline remains portable and reproducible.

The loading function uses exception handling for file-not-found, parsing, and
file I/O errors. Successful loading is logged together with the number of rows
and columns.

In [6]:
csv_path = Path("../part1_data_analytics/Metro_Interstate_Traffic_Volume.csv")

logger.debug(f"Raw CSV path: {csv_path.resolve()}")

In [7]:
csv_path.exists()

True

In [8]:
def load_raw_data(file_path):
    """Load the raw traffic CSV file safely."""
    try:
        df = pd.read_csv(file_path)

        logger.info(
            "Raw CSV loaded successfully: %d rows, %d columns.",
            df.shape[0],
            df.shape[1]
        )

        return df

    except FileNotFoundError:
        logger.error(
            "Raw CSV file was not found: %s",
            file_path,
            exc_info=True
        )
        return None

    except pd.errors.ParserError:
        logger.error(
            "Failed to parse the raw CSV file: %s",
            file_path,
            exc_info=True
        )
        return None

    except OSError:
        logger.error(
            "File I/O error while reading: %s",
            file_path,
            exc_info=True
        )
        return None

In [9]:
traffic_df = load_raw_data(csv_path)

2026-10-06 05:13:13,012 - INFO - __main__ - Raw CSV loaded successfully: 48204 rows, 9 columns.


### Task 1.4 – Validate the Data Schema

Before cleaning or transforming the dataset, the source schema is validated
against the nine expected columns.

If any required column is missing, the validation fails and an error is recorded
in the pipeline log.

In [10]:
EXPECTED_COLUMNS = [
    "holiday",
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "weather_main",
    "weather_description",
    "date_time",
    "traffic_volume"
]

In [11]:
def validate_schema(df, expected_columns):
    """Validate that all required columns exist in the dataset."""
    missing_columns = [
        column for column in expected_columns
        if column not in df.columns
    ]

    if missing_columns:
        logger.error(
            "Schema validation failed. Missing columns: %s",
            missing_columns
        )
        return False

    logger.info(
        "Schema validation successful. All %d expected columns are present.",
        len(expected_columns)
    )
    return True

In [12]:
schema_valid = validate_schema(traffic_df, EXPECTED_COLUMNS)

if not schema_valid:
    raise ValueError(
        "Pipeline stopped because schema validation failed."
    )

schema_valid

2026-10-06 05:13:13,048 - INFO - __main__ - Schema validation successful. All 9 expected columns are present.


True

### Task 1.5 – Clean and Validate the Data

The raw dataset is examined for inconsistent categorical values, invalid
date/time values, duplicate rows, physically impossible measurements, and
outliers.

Each cleaning action is logged separately so that modifications to the source
data can be traced and reproduced.

#### Task 1.5.1 – Standardise Categorical Values

Categorical columns are inspected for inconsistent spelling, capitalisation,
leading or trailing spaces, and other formatting differences before
standardisation is applied.

In [13]:
categorical_columns = [
    "holiday",
    "weather_main",
    "weather_description"
]

for column in categorical_columns:
    logger.info(
        "Unique values in %s: %d",
        column,
        traffic_df[column].nunique(dropna=False)
    )
    logger.debug(
        "%s values: %s",
        column,
        sorted(traffic_df[column].dropna().astype(str).unique())
    )

2026-10-06 05:13:13,063 - INFO - __main__ - Unique values in holiday: 12
2026-10-06 05:13:13,073 - INFO - __main__ - Unique values in weather_main: 11
2026-10-06 05:13:13,089 - INFO - __main__ - Unique values in weather_description: 38


In [14]:
holiday_values = sorted(traffic_df["holiday"].dropna().astype(str).unique())
weather_values = sorted(traffic_df["weather_main"].dropna().astype(str).unique())

holiday_values, weather_values

(['Christmas Day',
  'Columbus Day',
  'Independence Day',
  'Labor Day',
  'Martin Luther King Jr Day',
  'Memorial Day',
  'New Years Day',
  'State Fair',
  'Thanksgiving Day',
  'Veterans Day',
  'Washingtons Birthday'],
 ['Clear',
  'Clouds',
  'Drizzle',
  'Fog',
  'Haze',
  'Mist',
  'Rain',
  'Smoke',
  'Snow',
  'Squall',
  'Thunderstorm'])

**Finding:** The inspected categorical values were already largely consistent.
To make the pipeline robust to future input data, leading and trailing
whitespace is removed from all categorical columns. Existing category names
and capitalisation are otherwise preserved.

In [15]:
def standardise_categorical_values(df, columns):
    """Standardise categorical text fields without changing valid category names."""
    df = df.copy()

    for column in columns:
        before = df[column].copy()

        df[column] = df[column].apply(
            lambda value: value.strip() if isinstance(value, str) else value
        )

        # Treat NaN before and after as unchanged
        changed_mask = ~(
            before.eq(df[column]) |
            (before.isna() & df[column].isna())
        )
        changed_rows = changed_mask.sum()

        if changed_rows > 0:
            logger.warning(
                "%d rows modified while standardising categorical column '%s'.",
                changed_rows,
                column
            )
        else:
            logger.info(
                "Categorical column '%s' required no standardisation changes.",
                column
            )

    return df

In [16]:
traffic_df = standardise_categorical_values(
    traffic_df,
    categorical_columns
)

2026-10-06 05:13:13,191 - INFO - __main__ - Categorical column 'holiday' required no standardisation changes.
2026-10-06 05:13:13,217 - INFO - __main__ - Categorical column 'weather_main' required no standardisation changes.
2026-10-06 05:13:13,255 - INFO - __main__ - Categorical column 'weather_description' required no standardisation changes.


**Result:** The categorical standardisation check found no values requiring
modification in `holiday`, `weather_main`, or `weather_description`.
The pipeline nevertheless retains the standardisation step so that future input
data containing leading or trailing whitespace can be handled consistently.

#### Task 1.5.2 – Parse and Validate Date/Time

The `date_time` field is converted to a Pandas datetime data type using
validation that identifies values that cannot be parsed.

Invalid date/time values are logged before any affected rows are removed or
modified.

In [17]:
def parse_datetime_column(df, column="date_time"):
    """Parse and validate the date/time column."""
    df = df.copy()

    parsed_dates = pd.to_datetime(
        df[column],
        errors="coerce"
    )

    invalid_mask = parsed_dates.isna() & df[column].notna()
    invalid_count = invalid_mask.sum()

    if invalid_count > 0:
        logger.warning(
            "%d invalid date/time values detected in '%s'.",
            invalid_count,
            column
        )
    else:
        logger.info(
            "Date/time validation successful for '%s'; no invalid values detected.",
            column
        )

    df[column] = parsed_dates

    return df

In [18]:
traffic_df = parse_datetime_column(traffic_df)

2026-10-06 05:13:13,349 - INFO - __main__ - Date/time validation successful for 'date_time'; no invalid values detected.


In [19]:
traffic_df["date_time"].dtype

dtype('<M8[ns]')

**Result:** All `date_time` values were successfully parsed and validated.
No invalid date/time values were detected, and the column was converted to
Pandas `datetime64[ns]` format.

#### Task 1.5.3 – Identify and Remove Duplicate Rows

The dataset is checked for fully duplicated records. Duplicate rows can bias
traffic statistics and machine-learning models, so any duplicates detected are
removed and the number of affected rows is recorded in the log.

In [20]:
def remove_duplicates(df):
    """Identify and remove fully duplicated rows."""
    df = df.copy()

    duplicate_count = df.duplicated().sum()

    if duplicate_count > 0:
        df = df.drop_duplicates().reset_index(drop=True)

        logger.warning(
            "%d duplicate rows detected and removed.",
            duplicate_count
        )
    else:
        logger.info(
            "Duplicate check completed; no duplicate rows detected."
        )

    return df

In [21]:
traffic_df = remove_duplicates(traffic_df)

2026-10-06 05:13:13,521 - WARNING - __main__ - 17 duplicate rows detected and removed.


In [22]:
traffic_df.shape

(48187, 9)

**Result:** The duplicate check identified 17 fully duplicated rows.
These records were removed to prevent duplicate observations from biasing
subsequent analysis and machine-learning results. The removal was recorded
as a WARNING in the pipeline log.

#### Task 1.5.4 – Detect Impossible Values and Outliers

Numerical sensor fields are examined for physically impossible or implausible
values. Particular attention is given to temperature readings of 0 Kelvin and
extreme rainfall values, as specified in the assignment.

Invalid observations are identified before any imputation is performed so that
the number and reason for each modification can be logged clearly.

In [23]:
zero_kelvin_count = (traffic_df["temp"] == 0).sum()
max_rainfall = traffic_df["rain_1h"].max()

zero_kelvin_count, max_rainfall

(np.int64(10), 9831.3)

In [24]:
extreme_rain_count = (traffic_df["rain_1h"] > 9000).sum()

logger.info(
    "Impossible-value check: %d zero-Kelvin temperature records and %d rainfall records above 9000 mm detected.",
    zero_kelvin_count,
    extreme_rain_count
)

zero_kelvin_count, extreme_rain_count

2026-10-06 05:13:13,573 - INFO - __main__ - Impossible-value check: 10 zero-Kelvin temperature records and 1 rainfall records above 9000 mm detected.


(np.int64(10), np.int64(1))

**Result:** The impossible-value check identified 10 temperature records at
0 Kelvin and 1 rainfall record above 9,000 mm. The maximum recorded rainfall
was 9,831.3 mm.

#### Task 1.5.5 – Impute Invalid Values

Invalid 0 Kelvin temperature readings are replaced using the median valid
temperature for the corresponding calendar month. This preserves seasonal
temperature differences rather than applying a single global value.

Rainfall values above 9,000 mm are treated as implausible sensor readings and
replaced using the median of the valid rainfall observations. Each imputation
is recorded as a WARNING in the pipeline log.

In [25]:
def impute_zero_kelvin_by_month(df):
    """Replace 0 K temperatures with the median valid temperature for each month."""
    df = df.copy()

    zero_mask = df["temp"] == 0
    affected_rows = zero_mask.sum()

    if affected_rows == 0:
        logger.info("No zero-Kelvin temperature values required imputation.")
        return df

    for month in sorted(df.loc[zero_mask, "date_time"].dt.month.unique()):
        month_zero_mask = zero_mask & (df["date_time"].dt.month == month)

        valid_month_values = df.loc[
            (df["date_time"].dt.month == month) & (df["temp"] > 0),
            "temp"
        ]

        monthly_median = valid_month_values.median()

        if pd.isna(monthly_median):
            monthly_median = df.loc[df["temp"] > 0, "temp"].median()

            logger.warning(
                "No valid monthly temperature values found for month %d; "
                "global valid temperature median used instead.",
                month
            )

        logger.debug(
            "Month %d valid temperature median used for imputation: %.2f K.",
            month,
            monthly_median
        )

        df.loc[month_zero_mask, "temp"] = monthly_median

    logger.warning(
        "%d zero-Kelvin temperature records imputed using monthly median temperatures.",
        affected_rows
    )

    return df

In [26]:
traffic_df = impute_zero_kelvin_by_month(traffic_df)

2026-10-06 05:13:13,634 - WARNING - __main__ - 10 zero-Kelvin temperature records imputed using monthly median temperatures.


In [27]:
(traffic_df["temp"] == 0).sum()

np.int64(0)

In [28]:
def impute_extreme_rainfall(df, threshold=9000):
    """Replace implausible rainfall values with the median valid rainfall."""
    df = df.copy()

    extreme_mask = df["rain_1h"] > threshold
    affected_rows = extreme_mask.sum()

    if affected_rows == 0:
        logger.info(
            "No rainfall values above %.0f mm required imputation.",
            threshold
        )
        return df

    valid_rainfall = df.loc[
        df["rain_1h"] <= threshold,
        "rain_1h"
    ]

    rainfall_median = valid_rainfall.median()

    logger.debug(
        "Valid rainfall median used for imputation: %.2f mm.",
        rainfall_median
    )

    df.loc[extreme_mask, "rain_1h"] = rainfall_median

    logger.warning(
        "%d rainfall record(s) above %.0f mm imputed using the valid rainfall median.",
        affected_rows,
        threshold
    )

    return df

In [29]:
traffic_df = impute_extreme_rainfall(traffic_df)

2026-10-06 05:13:13,678 - WARNING - __main__ - 1 rainfall record(s) above 9000 mm imputed using the valid rainfall median.


In [30]:
(traffic_df["rain_1h"] > 9000).sum()

np.int64(0)

#### Task 1.5.6 – End-to-End Cleaning Validation

The raw dataset is reloaded and all completed cleaning functions are applied
sequentially to confirm that the cleaning workflow produces consistent results
from the original source data.


In [31]:
traffic_df = load_raw_data(csv_path)

2026-10-06 05:13:13,826 - INFO - __main__ - Raw CSV loaded successfully: 48204 rows, 9 columns.


In [32]:
traffic_df = standardise_categorical_values(
    traffic_df,
    categorical_columns
)

traffic_df = parse_datetime_column(traffic_df)

traffic_df = remove_duplicates(traffic_df)

traffic_df = impute_zero_kelvin_by_month(traffic_df)

traffic_df = impute_extreme_rainfall(traffic_df)

2026-10-06 05:13:13,862 - INFO - __main__ - Categorical column 'holiday' required no standardisation changes.
2026-10-06 05:13:13,898 - INFO - __main__ - Categorical column 'weather_main' required no standardisation changes.
2026-10-06 05:13:13,938 - INFO - __main__ - Categorical column 'weather_description' required no standardisation changes.
2026-10-06 05:13:13,981 - INFO - __main__ - Date/time validation successful for 'date_time'; no invalid values detected.
2026-10-06 05:13:14,060 - WARNING - __main__ - 17 duplicate rows detected and removed.
2026-10-06 05:13:14,098 - WARNING - __main__ - 10 zero-Kelvin temperature records imputed using monthly median temperatures.
2026-10-06 05:13:14,108 - WARNING - __main__ - 1 rainfall record(s) above 9000 mm imputed using the valid rainfall median.


In [33]:
traffic_df.shape, \
(traffic_df["temp"] == 0).sum(), \
(traffic_df["rain_1h"] > 9000).sum(), \
traffic_df.duplicated().sum()

((48187, 9), np.int64(0), np.int64(0), np.int64(0))

**Result:** The final cleaning validation confirmed that the processed dataset
contains 48,187 rows and 9 columns. All 17 duplicate rows were removed,
all 10 zero-Kelvin temperature readings were imputed using monthly median
temperatures, and the single rainfall value above 9,000 mm was imputed using
the median of valid rainfall observations.

After cleaning, no duplicate rows, zero-Kelvin temperatures, or rainfall values
above 9,000 mm remained.

#### Task 1.5.7 – Final Validation and Save Cleaned Data

The cleaned dataset is validated before being written to disk. The final output
will be saved as a separate CSV file so that the original raw dataset remains
unchanged and the cleaned data can be reused in subsequent feature-engineering
tasks.

In [34]:
def final_validation(df):
    """Perform final validation checks on the cleaned dataset."""

    duplicate_count = df.duplicated().sum()
    zero_kelvin_count = (df["temp"] == 0).sum()
    extreme_rain_count = (df["rain_1h"] > 9000).sum()
    invalid_datetime_count = df["date_time"].isna().sum()

    if (
        duplicate_count == 0
        and zero_kelvin_count == 0
        and extreme_rain_count == 0
        and invalid_datetime_count == 0
    ):
        logger.info(
            "Final validation successful: %d rows, %d columns; "
            "no duplicates, zero-Kelvin temperatures, extreme rainfall values, "
            "or invalid date/time values remain.",
            df.shape[0],
            df.shape[1]
        )
        return True

    logger.error(
        "Final validation failed: duplicates=%d, zero_kelvin=%d, "
        "extreme_rainfall=%d, invalid_datetime=%d.",
        duplicate_count,
        zero_kelvin_count,
        extreme_rain_count,
        invalid_datetime_count
    )
    return False

In [35]:
validation_passed = final_validation(traffic_df)
validation_passed

2026-10-06 05:13:14,227 - INFO - __main__ - Final validation successful: 48187 rows, 9 columns; no duplicates, zero-Kelvin temperatures, extreme rainfall values, or invalid date/time values remain.


True

In [36]:
cleaned_output_path = Path("cleaned_traffic_data.csv")

try:
    traffic_df.to_csv(cleaned_output_path, index=False)

    logger.info(
        "Cleaned dataset saved successfully to '%s': %d rows, %d columns.",
        cleaned_output_path,
        traffic_df.shape[0],
        traffic_df.shape[1]
    )

except OSError:
    logger.error(
        "Failed to save cleaned dataset to '%s'.",
        cleaned_output_path,
        exc_info=True
    )

2026-10-06 05:13:14,762 - INFO - __main__ - Cleaned dataset saved successfully to 'cleaned_traffic_data.csv': 48187 rows, 9 columns.


In [37]:
cleaned_output_path.exists()

True

**Task 1 Result:** The cleaned dataset was successfully validated and saved as
`cleaned_traffic_data.csv`. The final dataset contains 48,187 rows and 9 columns,
with no duplicate rows, zero-Kelvin temperature readings, rainfall values above
9,000 mm, or invalid date/time values remaining.

The tested workflow will now be consolidated into the required `pipeline.py`
script for reproducible end-to-end execution.